In [1]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----
# install.packages("\")
# # install.packages("dplyr")
# # install.packages("tidyr")

# install.packages("vegan")
library(dplyr)

library(readxl)
library(readr)
library(RNetCDF)

# base settings
# -----
conf_vlab_name     <- "BlueCarbon"
# conf_workflow_name <- "PEMA"
conf_vlab_version  <- "v2"

# conf_workflow_id   <- f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
param_workflow_name <- "workflow name"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# dir_code <- file.path("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, conf_vlab_version, "Git public", "library")
# if (!dir.exists(dir_code)) {dir.create(dir_code, recursive=TRUE)}

# dir_data <- file.path("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, conf_vlab_version, param_workflow_name)
# if (!dir.exists(dir_data)) {dir.create(dir_data, recursive=TRUE)}

# local
# -----
conf_dir_workspace <- file.path("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp <- file.path("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      <- "naa-vre-public"
conf_minio_public_bucket_root <- paste(paste("vl", tolower(conf_vlab_name), sep="-"), tolower(conf_vlab_version), sep="/")
conf_minio_public_local_root  <- file.path(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  <- file.path(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  <- file.path(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data")

conf_minio_user_bucket        <- "naa-vre-user-data"
# conf_minio_user_bucket_root   <- param_user_email
conf_minio_user_bucket_root   <- conf_vlab_name
conf_minio_user_local_root    <- file.path(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    <- file.path(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    <- file.path(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   param_workflow_name)
conf_minio_user_local_flog    <- file.path(conf_minio_user_local_data, "log.md")

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# Input param
# -----
conf_delimiter_tsv <- "\t"
conf_delimiter_csv <- ","

# GUI, File
# .....
param_file <- "PercOC_only_unique_reduced_df_predictions.csv"
# param_latitude <- 56.0953
# param_longitude <- 14.2785
# param_seagrass_species <- "Posidonia oceanica"

# Output param
# -----
# results
conf_file_result_txt <- "Seagrass_site_result.txt"
conf_file_result_csv <- "Seagrass_site_result.csv"

conf_tmp_file_SG_modeling_dataframe <- "SG_modeling_dataframe.csv"
conf_tmp_file_result <- "tmp_result.rds"

print("Finish: NaaVRE parameters")
print("Workspace public:")
print(paste("  Root", conf_minio_public_local_root, sep=": "))
print(paste("  Code", conf_minio_public_local_code, sep=": "))
print(paste("  Data", conf_minio_public_local_data, sep=": "))

print("Workspace user:")
print(paste("  Root", conf_minio_user_local_root, sep=": "))
print(paste("  Code", conf_minio_user_local_code, sep=": "))
print(paste("  Data", conf_minio_user_local_data, sep=": "))
print(paste("  Log",  conf_minio_user_local_flog, sep=":  "))



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[1] "Finish: NaaVRE parameters"
[1] "Workspace public:"
[1] "  Root: //home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2"
[1] "  Code: //home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/code"
[1] "  Data: //home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data"
[1] "Workspace user:"
[1] "  Root: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon"
[1] "  Code: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/library"
[1] "  Data: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name"
[1] "  Log:  //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/log.md"


In [2]:
# User configuration
RASTER_BASE <- file.path(conf_minio_public_local_data)
MODEL_PATH <- file.path(conf_minio_public_local_data, "GPR_final.rds")
INPUT_PATH <- file.path(conf_minio_user_local_data, param_file)

str_time <- format(Sys.time(),  "%Y%m%d_%H%M%S")
fname_result_csv <- paste(str_time, conf_file_result_csv, sep ="-")
fname_result_txt <- paste(str_time, conf_file_result_txt, sep ="-")
OUTPUT_PATH_CSV <- file.path(conf_minio_user_local_data, fname_result_csv)
OUTPUT_PATH_TXT <- file.path(conf_minio_user_local_data, fname_result_txt)

POINT_COLUMNS <- c("longitude", "latitude", "seagrass_species")
USE_CLOSEST_RASTER_VALUE <- TRUE
RASTER_METHOD <- "nearest" # "nearest" or "bilinear"
RETURN_PREDICTION_SE <- TRUE
PREDICTION_MEAN_COLUMN <- "predicted_carbon_density"
PREDICTION_SE_COLUMN <- "predicted_se"
RS_COVARIATES <- c(
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.CHL.chlor_a.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.IOP.adg_443.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.IOP.bbp_443.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_443.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_620.4km"
)

cat("RASTER_BASE:", RASTER_BASE, "\n")
cat("MODEL_PATH:", MODEL_PATH, "\n")
cat("INPUT_PATH:", INPUT_PATH, "\n")
cat("OUTPUT_PATH_CSV:", OUTPUT_PATH_CSV, "\n")
cat("OUTPUT_PATH_TXT:", OUTPUT_PATH_TXT, "\n")
if (!file.exists(INPUT_PATH)) stop("Input file not found: ", INPUT_PATH, call. = FALSE)
if (!file.exists(MODEL_PATH)) stop("GPR model file not found: ", MODEL_PATH, call. = FALSE)
if (!dir.exists(RASTER_BASE)) stop("Raster directory not found: ", RASTER_BASE, call. = FALSE)

RASTER_BASE: //home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data 
MODEL_PATH: //home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/GPR_final.rds 
INPUT_PATH: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/PercOC_only_unique_reduced_df_predictions.csv 
OUTPUT_PATH_CSV: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/20261027_135713-Seagrass_site_result.csv 
OUTPUT_PATH_TXT: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/20261027_135713-Seagrass_site_result.txt 


In [3]:
# Load input points
if (grepl("\\.xlsx?$", INPUT_PATH, ignore.case = TRUE)) {
  if (!requireNamespace("readxl", quietly = TRUE)) {
    stop("Package 'readxl' is required for Excel input.", call. = FALSE)
  }
  input_data <- readxl::read_excel(INPUT_PATH, sheet = 1)
} else if (grepl("\\.csv$", INPUT_PATH, ignore.case = TRUE)) {
  input_data <- readr::read_csv(INPUT_PATH, show_col_types = FALSE)
} else {
  stop("Input must be a .csv, .xls, or .xlsx file: ", INPUT_PATH, call. = FALSE)
}

# Load the saved GPR model.
model <- readRDS(MODEL_PATH)
if (!identical(model$model_type, "GPR") && !inherits(model$model, "GauPro")) {
  stop("The model file does not contain a GPR model: ", MODEL_PATH, call. = FALSE)
}
PREDICTOR_VARS <- model$predictor_vars
if (is.null(PREDICTOR_VARS) || length(PREDICTOR_VARS) == 0L) {
  stop("The GPR model does not contain predictor_vars.", call. = FALSE)
}
MISSING_COLUMNS <- setdiff(POINT_COLUMNS, names(input_data))
if (length(MISSING_COLUMNS) > 0L) {
  stop(
    "Input data is missing required column(s): ",
    paste(MISSING_COLUMNS, collapse = ", "),
    call. = FALSE
  )
}
POINT_DATA_COLUMNS <- unique(c(
  POINT_COLUMNS,
  intersect(PREDICTOR_VARS, names(input_data))
))
POINT_DATA <- as.data.frame(input_data[, POINT_DATA_COLUMNS, drop = FALSE])
RASTER_COVARIATES <- setdiff(PREDICTOR_VARS, names(POINT_DATA))
cat("Input points:", nrow(input_data), " | Model predictors:", length(PREDICTOR_VARS), "\n")
cat("Raster covariates to extract:", length(RASTER_COVARIATES), "\n")
if (length(RASTER_COVARIATES) > 0L) {
  RASTER_PACKAGES <- c("RNetCDF", "terra")
  MISSING_PACKAGES <- RASTER_PACKAGES[
    !vapply(RASTER_PACKAGES, requireNamespace, logical(1), quietly = TRUE)
  ]
  if (length(MISSING_PACKAGES) > 0L) {
    stop(
      "Missing required package(s) for raster extraction: ",
      paste(MISSING_PACKAGES, collapse = ", "),
      ". Restore the v2 renv environment or install these packages.",
      call. = FALSE
    )
  }
}

Input points: 336  | Model predictors: 14 
Raster covariates to extract: 13 


In [4]:
# Match each required predictor to a NetCDF file and its 2D latitude/longitude variable
RASTER_CONFIG <- list()
REQUESTED_COVARIATES <- unique(tolower(RASTER_COVARIATES))
if (length(REQUESTED_COVARIATES) > 0L) {
  if (!dir.exists(RASTER_BASE)) stop("Raster directory not found: ", RASTER_BASE, call. = FALSE)
  RASTER_FILES <- list.files(
    RASTER_BASE, pattern = "[.]nc$", full.names = TRUE, ignore.case = TRUE
  )
  RASTER_FILE_NAMES <- tolower(gsub(
    "-", "_", sub("[.]nc$", "", basename(RASTER_FILES), ignore.case = TRUE), fixed = TRUE
  ))
  if (anyDuplicated(RASTER_FILE_NAMES)) {
    stop("Raster filenames are ambiguous after normalization in: ", RASTER_BASE, call. = FALSE)
  }
  MISSING_RASTERS <- setdiff(REQUESTED_COVARIATES, RASTER_FILE_NAMES)
  if (length(MISSING_RASTERS) > 0L) {
    stop(
      "No NetCDF raster found for required covariate(s): ",
      paste(MISSING_RASTERS, collapse = ", "),
      call. = FALSE
    )
  }

  # Loop through each requested covariate and find its corresponding NetCDF file and 2D lat/lon variable
  for (COVARIATE in REQUESTED_COVARIATES) {
    RASTER_FILE <- RASTER_FILES[match(COVARIATE, RASTER_FILE_NAMES)]

    # Open the NetCDF file
    NC_HANDLE <- RNetCDF::open.nc(RASTER_FILE)
    NC_VARIABLE_NAME <- NULL
    NC_VARIABLE_COUNT <- RNetCDF::file.inq.nc(NC_HANDLE)$nvars
    for (NC_INDEX in seq_len(NC_VARIABLE_COUNT) - 1L) {
      NC_VARIABLE <- RNetCDF::var.inq.nc(NC_HANDLE, NC_INDEX)
      if (length(NC_VARIABLE$dimids) != 2L) next
      
      NC_DIMENSION_NAMES <- character()
      for (NC_DIMENSION_ID in NC_VARIABLE$dimids) {
        NC_DIMENSION_NAMES <- c(
          NC_DIMENSION_NAMES,
          RNetCDF::dim.inq.nc(NC_HANDLE, NC_DIMENSION_ID)$name
        )
      }
      if (any(grepl("lat", NC_DIMENSION_NAMES, ignore.case = TRUE)) &&
          any(grepl("lon", NC_DIMENSION_NAMES, ignore.case = TRUE))) {
        NC_VARIABLE_NAME <- NC_VARIABLE$name
        break
      }
    }
    RNetCDF::close.nc(NC_HANDLE)

    # Store the raster configuration for the current covariate
    if (is.null(NC_VARIABLE_NAME)) {
      stop(
        "No 2D latitude/longitude variable found in raster: ", basename(RASTER_FILE),
        call. = FALSE
      )
    }
    RASTER_CONFIG[[COVARIATE]] <- list(
      file = file.path(RASTER_BASE, basename(RASTER_FILE)),
      varname = NC_VARIABLE_NAME
    )
  }
}
# Return the final raster configuration for all requested covariates
RASTER_CONFIG

$surf_fgco2_mean_molc_m2_yr
$surf_fgco2_mean_molc_m2_yr$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/Surf_fgco2_mean_molC_m2_yr.nc"

$surf_fgco2_mean_molc_m2_yr$varname
[1] "Surf_fgco2_mean_molC_m2_yr"


$s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km
$s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.IOP.adg_443.4km.nc"

$s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km$varname
[1] "adg_443"


$vo_mean_1.5m_m_s
$vo_mean_1.5m_m_s$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/vo_mean_1.5m_m_s.nc"

$vo_mean_1.5m_m_s$varname
[1] "vo_mean_1.5m_m_s"


$uo_p90_1.5m_m_s
$uo_p90_1.5m_m_s$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/uo_p90_1.5m_m_s.nc"

$uo_p90_1.5m_m_s$varname
[1] "uo_p90_1.5m_m_s"


$vo_p90_1.5m_m_s
$vo_p90_1.5m_m_s$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/vo_p90_1.5m_m_s.nc"

$vo_p90_1.5m_m_s$varname
[1] "vo_p90_1.5m_m_s"


$sw_ke_mean_cm2_s2
$sw_ke_mean_cm2_s2$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/SW_KE_mean_cm2_s2.nc"

$sw_ke_mean_cm2_s2$varname
[1] "SW_KE_mean_cm2_s2"


$daily_std_wave_height_mean_m
$daily_std_wave_height_mean_m$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/Daily_std_wave_height_mean_m.nc"

$daily_std_wave_height_mean_m$varname
[1] "Daily_std_wave_height_mean_m"


$fe_mean_monthly_1.5m_mmol_m3
$fe_mean_monthly_1.5m_mmol_m3$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/fe_mean_monthly_1.5m_mmol_m3.nc"

$fe_mean_monthly_1.5m_mmol_m3$varname
[1] "fe_mean_1.5m_mmol_m3"


$bottomt_p95_daily_c
$bottomt_p95_daily_c$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/bottomT_p95_daily_C.nc"

$bottomt_p95_daily_c$varname
[1] "p95_bottomT_daily_C"


$spco2_monthly_mean_pa
$spco2_monthly_mean_pa$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/spco2_monthly_mean_Pa.nc"

$spco2_monthly_mean_pa$varname
[1] "spco2_monthly_mean_Pa"


$s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km
$s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.CHL.chlor_a.4km.nc"

$s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km$varname
[1] "chlor_a"


$s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km
$s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_620.4km.nc"

$s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km$varname
[1] "Rrs_620"


$no_sig_wave_heights_mean
$no_sig_wave_heights_mean$file
[1] "//home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/No_sig_wave_heights_mean.nc"

$no_sig_wave_heights_mean$varname
[1] "No_sig_wave_heights_mean"

In [5]:
# Extract environmental covariates at each point
PREDICTION_DATA <- POINT_DATA
if (length(RASTER_COVARIATES) > 0L) {
  POINT_COORDINATES <- as.matrix(POINT_DATA[, c("longitude", "latitude"), drop = FALSE])
  for (COVARIATE in RASTER_COVARIATES) {
    COVARIATE_CONFIG <- RASTER_CONFIG[[tolower(COVARIATE)]]
    if (is.null(COVARIATE_CONFIG)) {
      stop("Raster configuration not found for covariate: ", COVARIATE, call. = FALSE)
    }

    # Construct the full path to the NetCDF raster file
    RASTER_PATH <- COVARIATE_CONFIG$file
    if (!file.exists(RASTER_PATH)) stop("NetCDF file not found: ", RASTER_PATH, call. = FALSE)

    RASTER <- terra::rast(RASTER_PATH)
    if (!is.null(COVARIATE_CONFIG$varname) &&
        nzchar(COVARIATE_CONFIG$varname) &&
        COVARIATE_CONFIG$varname %in% names(RASTER)) {
      RASTER <- RASTER[[COVARIATE_CONFIG$varname]]
    } else if (terra::nlyr(RASTER) > 1L) {
      warning(
        "Variable '", COVARIATE_CONFIG$varname, "' not found in ",
        COVARIATE_CONFIG$file, "; using first layer in file. Check the raster configuration."
      )
      RASTER <- RASTER[[1L]]
    }

    # Extract the raster values at the point locations
    TERRA_METHOD <- if (identical(tolower(RASTER_METHOD), "bilinear")) "bilinear" else "simple"
    COVARIATE_VALUES <- as.numeric(
      terra::extract(RASTER, POINT_COORDINATES, method = TERRA_METHOD)[, 1L]
    )
    if (USE_CLOSEST_RASTER_VALUE && anyNA(COVARIATE_VALUES) &&
        requireNamespace("FNN", quietly = TRUE)) {
      MISSING_POINT_INDICES <- which(is.na(COVARIATE_VALUES))
      RASTER_VALUES <- terra::values(RASTER, mat = FALSE)
      VALID_CELL_INDICES <- which(!is.na(RASTER_VALUES))
      
      # Find the nearest valid raster value for each missing point
      if (length(VALID_CELL_INDICES) > 0L) {
        NEAREST_VALID_INDICES <- FNN::get.knnx(
          data = terra::xyFromCell(RASTER, VALID_CELL_INDICES),
          query = POINT_COORDINATES[MISSING_POINT_INDICES, , drop = FALSE],
          k = 1L
        )$nn.index[, 1L]
        COVARIATE_VALUES[MISSING_POINT_INDICES] <-
          RASTER_VALUES[VALID_CELL_INDICES][NEAREST_VALID_INDICES]
      }
    }
    PREDICTION_DATA[[COVARIATE]] <- COVARIATE_VALUES
  }
}

# Match the non-negative preprocessing used for remote-sensing covariates.
EXISTING_RS_COVARIATES <- intersect(RS_COVARIATES, colnames(PREDICTION_DATA))
for (COVARIATE in EXISTING_RS_COVARIATES) {
  if (is.numeric(PREDICTION_DATA[[COVARIATE]])) {
    PREDICTION_DATA[[COVARIATE]] <- pmax(PREDICTION_DATA[[COVARIATE]], 0, na.rm = FALSE)
  }
}
# Identify any raster covariates for which extraction failed (i.e., contains NA values)
MISSING_EXTRACTED_COVARIATES <- character()
for (COVARIATE in RASTER_COVARIATES) {
  if (any(is.na(PREDICTION_DATA[[COVARIATE]]))) {
    MISSING_EXTRACTED_COVARIATES <- c(MISSING_EXTRACTED_COVARIATES, COVARIATE)
  }
}
if (length(MISSING_EXTRACTED_COVARIATES) > 0L) {
  warning(
    "Missing extracted covariate values: ",
    paste(MISSING_EXTRACTED_COVARIATES, collapse = ", "),
    call. = FALSE
  )
}

cat("Prepared point covariates:", nrow(PREDICTION_DATA), "rows x", ncol(PREDICTION_DATA), "columns\n")

Prepared point covariates: 336 rows x 16 columns


In [6]:
# Apply the saved model's categorical encoding and scaling
MISSING_PREDICTORS <- setdiff(PREDICTOR_VARS, names(PREDICTION_DATA))
if (length(MISSING_PREDICTORS) > 0L) {
  stop(
    "Prediction data is missing model predictor(s): ",
    paste(MISSING_PREDICTORS, collapse = ", "),
    call. = FALSE
  )
}
MODEL_COLUMNS <- if (is.null(model$encoded_names)) PREDICTOR_VARS else model$encoded_names
ENCODING <- model$encoding
ENCODING_TYPE <- if (is.null(ENCODING) || is.null(ENCODING$type)) NA_character_ else ENCODING$type
if (identical(ENCODING_TYPE, "one_hot") &&
    !is.null(model$species_means) &&
    !is.null(ENCODING$kernel_predictor_vars)) {
  ENCODING_TYPE <- "gpr_species_adjusted"
}
if (identical(ENCODING_TYPE, "gpr_species_adjusted")) {
  KERNEL_PREDICTORS <- if (is.null(ENCODING$kernel_predictor_vars)) {
    setdiff(PREDICTOR_VARS, "seagrass_species")
  } else {
    ENCODING$kernel_predictor_vars
  }
  X <- as.data.frame(PREDICTION_DATA[, KERNEL_PREDICTORS, drop = FALSE])
  ONE_HOT_PREDICTORS <- if (is.null(ENCODING$predictor_vars)) names(X) else ENCODING$predictor_vars
  X <- as.data.frame(X[, ONE_HOT_PREDICTORS, drop = FALSE])
  if (!is.null(ENCODING$levels) && length(ENCODING$levels) > 0L) {
    for (VARIABLE in names(ENCODING$levels)) {
      if (VARIABLE %in% names(X)) {
        X[[VARIABLE]] <- factor(
          as.character(X[[VARIABLE]]),
          levels = ENCODING$levels[[VARIABLE]]
        )
      }
    }
  }
  # Convert categorical variables to one-hot encoding
  X <- as.data.frame(model.matrix(~ . - 1, data = X))
  MISSING_MODEL_COLUMNS <- setdiff(MODEL_COLUMNS, names(X))
  for (VARIABLE in MISSING_MODEL_COLUMNS) X[[VARIABLE]] <- 0
  X <- X[, MODEL_COLUMNS, drop = FALSE]
} else if (identical(ENCODING_TYPE, "one_hot")) {
  X <- as.data.frame(PREDICTION_DATA[, PREDICTOR_VARS, drop = FALSE])
  ONE_HOT_PREDICTORS <- if (is.null(ENCODING$predictor_vars)) names(X) else ENCODING$predictor_vars

  # Subset the prediction data to only include the one-hot predictors
  X <- as.data.frame(X[, ONE_HOT_PREDICTORS, drop = FALSE])
  if (!is.null(ENCODING$levels) && length(ENCODING$levels) > 0L) {
    for (VARIABLE in names(ENCODING$levels)) {
      if (VARIABLE %in% names(X)) {
        X[[VARIABLE]] <- factor(
          as.character(X[[VARIABLE]]),
          levels = ENCODING$levels[[VARIABLE]]
        )
      }
    }
  }

  # Convert categorical variables to one-hot encoding
  X <- as.data.frame(model.matrix(~ . - 1, data = X))
  MISSING_MODEL_COLUMNS <- setdiff(MODEL_COLUMNS, names(X))
  for (VARIABLE in MISSING_MODEL_COLUMNS) X[[VARIABLE]] <- 0

  # Ensure the columns in the prediction data match the model columns
  X <- X[, MODEL_COLUMNS, drop = FALSE]
} else {
  X <- as.data.frame(PREDICTION_DATA[, PREDICTOR_VARS, drop = FALSE])
  if (!is.null(ENCODING) && !is.null(ENCODING$levels) && length(ENCODING$levels) > 0L) {
    for (VARIABLE in names(ENCODING$levels)) {
      if (VARIABLE %in% names(X)) {
        X[[VARIABLE]] <- as.integer(factor(
          as.character(X[[VARIABLE]]),
          levels = ENCODING$levels[[VARIABLE]]
        ))
        X[[VARIABLE]][is.na(X[[VARIABLE]])] <- 1L
      }
    }
  }
}

# Scale the prediction data using the model's scale parameters
SCALE_PARAMS <- model$scale_params
if (!is.null(SCALE_PARAMS) && is.null(SCALE_PARAMS$means) && !is.null(SCALE_PARAMS$x_center)) {
  SCALE_PARAMS <- list(means = SCALE_PARAMS$x_center, sds = SCALE_PARAMS$x_scale)
}
if (!is.null(SCALE_PARAMS) && length(SCALE_PARAMS$means) > 0L) {
  for (VARIABLE in names(SCALE_PARAMS$means)) {
    if (VARIABLE %in% names(X)) {
      X[[VARIABLE]] <- (X[[VARIABLE]] - SCALE_PARAMS$means[VARIABLE]) /
        SCALE_PARAMS$sds[VARIABLE]
    }
  }
}

# Convert the prediction data to a matrix suitable for GPR model input
X_MATRIX <- as.matrix(X[, MODEL_COLUMNS, drop = FALSE])
storage.mode(X_MATRIX) <- "double"
dim(X_MATRIX)

[1] 336  13

In [7]:
# Predict carbon density and optional standard error
cat("Predicting carbon density for", nrow(input_data), "point(s)...\n")
PREDICTION_OUTPUT <- model$model$pred(
  X_MATRIX,
  se.fit = RETURN_PREDICTION_SE,
  return_df = TRUE
)
if (is.list(PREDICTION_OUTPUT) && !is.null(PREDICTION_OUTPUT$mean)) {
  PREDICTION_MEAN <- as.numeric(PREDICTION_OUTPUT$mean)
  PREDICTION_SE <- if (RETURN_PREDICTION_SE) as.numeric(PREDICTION_OUTPUT$se) else NULL
} else {
  PREDICTION_MEAN <- as.numeric(PREDICTION_OUTPUT)
  PREDICTION_SE <- NULL
}

# Apply species-specific adjustment to the predicted carbon density if available
SPECIES_VARIABLE <- if (is.null(ENCODING) || is.null(ENCODING$species_var)) {
  NULL
} else {
  ENCODING$species_var
}
if (!is.null(model$species_means) &&
    !is.null(SPECIES_VARIABLE) &&
    SPECIES_VARIABLE %in% names(PREDICTION_DATA)) {
  SPECIES_FALLBACK <- if (is.null(model$global_mean)) 0 else model$global_mean
  SPECIES_ADJUSTMENT <- as.numeric(
    model$species_means[as.character(PREDICTION_DATA[[SPECIES_VARIABLE]])]
  )
  SPECIES_ADJUSTMENT[is.na(SPECIES_ADJUSTMENT)] <- SPECIES_FALLBACK
  PREDICTION_MEAN <- PREDICTION_MEAN + SPECIES_ADJUSTMENT
}

# Apply log transformation if the model was trained on log-transformed response
if (isTRUE(model$log_response)) {
  PREDICTION_MEAN <- exp(PREDICTION_MEAN)
  if (!is.null(PREDICTION_SE)) PREDICTION_SE <- PREDICTION_SE * PREDICTION_MEAN
}
if (length(PREDICTION_MEAN) != nrow(input_data)) {
  stop("GPR prediction count does not match the number of input rows.", call. = FALSE)
}

# Check that the number of predictions matches the number of input rows
input_data[[PREDICTION_MEAN_COLUMN]] <- PREDICTION_MEAN
if (!is.null(PREDICTION_SE)) input_data[[PREDICTION_SE_COLUMN]] <- PREDICTION_SE
head(input_data)

Predicting carbon density for 336 point(s)...


latitude,longitude,organic_carbon_percent,seagrass_species,random_core_variable,sediment_mean_depth_cm,median_carbon_density_100cm_calc,predicted_carbon_density,predicted_se,env_similarity_score,⋯,env_flag_extrapolation,carbon_stock,measured_carbon_density,measured_carbon_stock,density_residual,stock_residual,species_mean_predicted_carbon_density,species_mean_carbon_stock,species_mean_density_residual,species_mean_stock_residual
<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
55.29244,10.790408,0.18,Unspecified,Baltic Sea_Nyborg Fjord_NA_1998_7_NA_55.292436_10.790408_3,3.179415,0.0105525,0.009220519,0.007963927,0.5740222,⋯,FALSE,0.9521138,0.0105525,1.05525,1.031362e-03,0.103136219,0.01792956,1.792956,-0.007377055,-0.7377055
51.60265,4.119084,0.73,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_NA,3.179415,0.0068474,0.014461933,0.011716813,0.5754127,⋯,FALSE,0.9647075,0.0068474,0.68474,-2.799675e-03,-0.279967549,0.01792956,1.792956,-0.011082155,-1.1082155
51.60265,4.119084,0.30,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_1,3.179415,0.0028140,0.014461933,0.011716813,0.5754127,⋯,FALSE,0.9647075,0.0028140,0.28140,-6.833075e-03,-0.683307549,0.01792956,1.792956,-0.015115555,-1.5115555
44.65116,-1.096740,0.32,Unspecified,Bay of Biscay_Arcachon Bay_NA_1998_7_NA_44.651162_-1.09674_NA,3.179415,0.0100835,0.008615413,0.007718579,0.5813582,⋯,FALSE,0.8877507,0.0100835,1.00835,1.205993e-03,0.120599263,0.01792956,1.792956,-0.007846055,-0.7846055
34.70433,33.209667,0.70,Unspecified,Limassol Bay_LM_1_NA_1997_8_NA_34.7043333333_33.2096666667_5,3.179415,0.0065660,0.008583128,0.007689611,0.8417705,⋯,TRUE,0.8495970,0.0065660,0.65660,-1.929970e-03,-0.192997045,0.01792956,1.792956,-0.011363555,-1.1363555
34.70083,33.208833,0.90,Unspecified,Limassol Bay_LM_2_NA_1997_8_NA_34.7008333333_33.2088333333_15,3.179415,0.0084420,0.008583128,0.007689611,0.8417705,⋯,TRUE,0.8495970,0.0084420,0.84420,-5.397045e-05,-0.005397045,0.01792956,1.792956,-0.009487555,-0.9487555


In [8]:
# #####
cat("OUTPUT_PATH_CSV:", OUTPUT_PATH_CSV, "\n")
cat("OUTPUT_PATH_TXT:", OUTPUT_PATH_TXT, "\n")

# Carbon stock in the upper 30 and 100 cm
print("Carbon stock in the upper 30 and 100 cm")

# SG_modeling_dataframe <- readRDS(OUTPUT_PATH)
SG_modeling_dataframe <- input_data

# depths <- c(5, 15, 25, 35, 45, 55, 65, 75, 85, 95)

head(SG_modeling_dataframe)

# =====
# Calculate carbon stock per 10 cm
# Carbon stock (MgC/ha)=Carbon density (gC/cm³) × Depth (cm) × 100,000,000 (cm2/ha) ÷ 1,000,000 (g/Mg)
# Where:
#   1 hectare = 10,000 m² = 100,000,000 cm²
#   1 Mg = 1,000,000 g
SG_modeling_dataframe$carbon_stock_Mg_ha_upper30cm  = SG_modeling_dataframe$predicted_carbon_density* 3*10*100000000*(1/1000000)
SG_modeling_dataframe$carbon_stock_Mg_ha_upper100cm = SG_modeling_dataframe$predicted_carbon_density*10*10*100000000*(1/1000000)
SG_modeling_dataframe$carbon_stock_Mg_ha_upper30cm_se  = SG_modeling_dataframe$predicted_se* 3*10*100000000*(1/1000000)
SG_modeling_dataframe$carbon_stock_Mg_ha_upper100cm_se = SG_modeling_dataframe$predicted_se*10*10*100000000*(1/1000000)
head(SG_modeling_dataframe)

# Create sample_ID column
SG_modeling_dataframe$sample_ID <- rep(1:(nrow(SG_modeling_dataframe)/1), each = 1)
head(SG_modeling_dataframe)

# # #####
# # Create summary sentences

summary_dataframe <- SG_modeling_dataframe %>% 
  select(-c("sediment_mean_depth_cm",
            "random_core_variable",
            "predicted_carbon_density",
            "predicted_se"))

summary_sentences <- SG_modeling_dataframe %>%
  mutate(summary = paste0(
    "For the seagrass bed at latitude ", latitude, " and longitude ", longitude, "\n",
    "The species identity ", seagrass_species, "\n",
    "The predicted carbon stock in the upper \n",
    "  - upper 30cm  of the sediment is ", format(carbon_stock_Mg_ha_upper30cm,  digits=3, nsmall=3), " Mg/ha\n",
    "  - upper 100cm of the sediment is ", format(carbon_stock_Mg_ha_upper100cm, digits=3, nsmall=3), " Mg/ha\n",
    "The predicted carbon stock standard error in the upper \n",
    "  - upper 30cm  of the sediment is ", format(carbon_stock_Mg_ha_upper30cm_se,  digits=3, nsmall=3), " Mg/ha\n",
    "  - upper 100cm of the sediment is ", format(carbon_stock_Mg_ha_upper100cm_se, digits=3, nsmall=3), " Mg/ha\n"
  )) %>%
  pull(summary)

# Save result
# -----
# summary_dataframe
# summary(summary_dataframe)
write.csv(summary_dataframe, OUTPUT_PATH_CSV)
# minio_status <- put_object(bucket=conf_minio_user_bucket, file=file_result_csv, object=paste(param_user_email, fcloud_result_csv, sep="/"))

# summary_sentences
# cat(summary_sentences, sep = "\n")
cat(summary_sentences, sep = "\n", file = OUTPUT_PATH_TXT)
# minio_status <- put_object(bucket=conf_minio_user_bucket, file=file_result_txt, object=paste(param_user_email, fcloud_result_txt, sep="/"))

print("Carbon predicted")

OUTPUT_PATH_CSV: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/20261027_135713-Seagrass_site_result.csv 
OUTPUT_PATH_TXT: //home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/20261027_135713-Seagrass_site_result.txt 
[1] "Carbon stock in the upper 30 and 100 cm"


latitude,longitude,organic_carbon_percent,seagrass_species,random_core_variable,sediment_mean_depth_cm,median_carbon_density_100cm_calc,predicted_carbon_density,predicted_se,env_similarity_score,⋯,env_flag_extrapolation,carbon_stock,measured_carbon_density,measured_carbon_stock,density_residual,stock_residual,species_mean_predicted_carbon_density,species_mean_carbon_stock,species_mean_density_residual,species_mean_stock_residual
<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
55.29244,10.790408,0.18,Unspecified,Baltic Sea_Nyborg Fjord_NA_1998_7_NA_55.292436_10.790408_3,3.179415,0.0105525,0.009220519,0.007963927,0.5740222,⋯,FALSE,0.9521138,0.0105525,1.05525,1.031362e-03,0.103136219,0.01792956,1.792956,-0.007377055,-0.7377055
51.60265,4.119084,0.73,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_NA,3.179415,0.0068474,0.014461933,0.011716813,0.5754127,⋯,FALSE,0.9647075,0.0068474,0.68474,-2.799675e-03,-0.279967549,0.01792956,1.792956,-0.011082155,-1.1082155
51.60265,4.119084,0.30,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_1,3.179415,0.0028140,0.014461933,0.011716813,0.5754127,⋯,FALSE,0.9647075,0.0028140,0.28140,-6.833075e-03,-0.683307549,0.01792956,1.792956,-0.015115555,-1.5115555
44.65116,-1.096740,0.32,Unspecified,Bay of Biscay_Arcachon Bay_NA_1998_7_NA_44.651162_-1.09674_NA,3.179415,0.0100835,0.008615413,0.007718579,0.5813582,⋯,FALSE,0.8877507,0.0100835,1.00835,1.205993e-03,0.120599263,0.01792956,1.792956,-0.007846055,-0.7846055
34.70433,33.209667,0.70,Unspecified,Limassol Bay_LM_1_NA_1997_8_NA_34.7043333333_33.2096666667_5,3.179415,0.0065660,0.008583128,0.007689611,0.8417705,⋯,TRUE,0.8495970,0.0065660,0.65660,-1.929970e-03,-0.192997045,0.01792956,1.792956,-0.011363555,-1.1363555
34.70083,33.208833,0.90,Unspecified,Limassol Bay_LM_2_NA_1997_8_NA_34.7008333333_33.2088333333_15,3.179415,0.0084420,0.008583128,0.007689611,0.8417705,⋯,TRUE,0.8495970,0.0084420,0.84420,-5.397045e-05,-0.005397045,0.01792956,1.792956,-0.009487555,-0.9487555


latitude,longitude,organic_carbon_percent,seagrass_species,random_core_variable,sediment_mean_depth_cm,median_carbon_density_100cm_calc,predicted_carbon_density,predicted_se,env_similarity_score,⋯,density_residual,stock_residual,species_mean_predicted_carbon_density,species_mean_carbon_stock,species_mean_density_residual,species_mean_stock_residual,carbon_stock_Mg_ha_upper30cm,carbon_stock_Mg_ha_upper100cm,carbon_stock_Mg_ha_upper30cm_se,carbon_stock_Mg_ha_upper100cm_se
<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
55.29244,10.790408,0.18,Unspecified,Baltic Sea_Nyborg Fjord_NA_1998_7_NA_55.292436_10.790408_3,3.179415,0.0105525,0.009220519,0.007963927,0.5740222,⋯,1.031362e-03,0.103136219,0.01792956,1.792956,-0.007377055,-0.7377055,27.66156,92.20519,23.89178,79.63927
51.60265,4.119084,0.73,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_NA,3.179415,0.0068474,0.014461933,0.011716813,0.5754127,⋯,-2.799675e-03,-0.279967549,0.01792956,1.792956,-0.011082155,-1.1082155,43.38580,144.61933,35.15044,117.16813
51.60265,4.119084,0.30,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_1,3.179415,0.0028140,0.014461933,0.011716813,0.5754127,⋯,-6.833075e-03,-0.683307549,0.01792956,1.792956,-0.015115555,-1.5115555,43.38580,144.61933,35.15044,117.16813
44.65116,-1.096740,0.32,Unspecified,Bay of Biscay_Arcachon Bay_NA_1998_7_NA_44.651162_-1.09674_NA,3.179415,0.0100835,0.008615413,0.007718579,0.5813582,⋯,1.205993e-03,0.120599263,0.01792956,1.792956,-0.007846055,-0.7846055,25.84624,86.15413,23.15574,77.18579
34.70433,33.209667,0.70,Unspecified,Limassol Bay_LM_1_NA_1997_8_NA_34.7043333333_33.2096666667_5,3.179415,0.0065660,0.008583128,0.007689611,0.8417705,⋯,-1.929970e-03,-0.192997045,0.01792956,1.792956,-0.011363555,-1.1363555,25.74938,85.83128,23.06883,76.89611
34.70083,33.208833,0.90,Unspecified,Limassol Bay_LM_2_NA_1997_8_NA_34.7008333333_33.2088333333_15,3.179415,0.0084420,0.008583128,0.007689611,0.8417705,⋯,-5.397045e-05,-0.005397045,0.01792956,1.792956,-0.009487555,-0.9487555,25.74938,85.83128,23.06883,76.89611


latitude,longitude,organic_carbon_percent,seagrass_species,random_core_variable,sediment_mean_depth_cm,median_carbon_density_100cm_calc,predicted_carbon_density,predicted_se,env_similarity_score,⋯,stock_residual,species_mean_predicted_carbon_density,species_mean_carbon_stock,species_mean_density_residual,species_mean_stock_residual,carbon_stock_Mg_ha_upper30cm,carbon_stock_Mg_ha_upper100cm,carbon_stock_Mg_ha_upper30cm_se,carbon_stock_Mg_ha_upper100cm_se,sample_ID
<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
55.29244,10.790408,0.18,Unspecified,Baltic Sea_Nyborg Fjord_NA_1998_7_NA_55.292436_10.790408_3,3.179415,0.0105525,0.009220519,0.007963927,0.5740222,⋯,0.103136219,0.01792956,1.792956,-0.007377055,-0.7377055,27.66156,92.20519,23.89178,79.63927,1
51.60265,4.119084,0.73,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_NA,3.179415,0.0068474,0.014461933,0.011716813,0.5754127,⋯,-0.279967549,0.01792956,1.792956,-0.011082155,-1.1082155,43.38580,144.61933,35.15044,117.16813,2
51.60265,4.119084,0.30,Unspecified,North Sea_Eastern Scheld_NA_1998_8_NA_51.602651_4.119084_1,3.179415,0.0028140,0.014461933,0.011716813,0.5754127,⋯,-0.683307549,0.01792956,1.792956,-0.015115555,-1.5115555,43.38580,144.61933,35.15044,117.16813,3
44.65116,-1.096740,0.32,Unspecified,Bay of Biscay_Arcachon Bay_NA_1998_7_NA_44.651162_-1.09674_NA,3.179415,0.0100835,0.008615413,0.007718579,0.5813582,⋯,0.120599263,0.01792956,1.792956,-0.007846055,-0.7846055,25.84624,86.15413,23.15574,77.18579,4
34.70433,33.209667,0.70,Unspecified,Limassol Bay_LM_1_NA_1997_8_NA_34.7043333333_33.2096666667_5,3.179415,0.0065660,0.008583128,0.007689611,0.8417705,⋯,-0.192997045,0.01792956,1.792956,-0.011363555,-1.1363555,25.74938,85.83128,23.06883,76.89611,5
34.70083,33.208833,0.90,Unspecified,Limassol Bay_LM_2_NA_1997_8_NA_34.7008333333_33.2088333333_15,3.179415,0.0084420,0.008583128,0.007689611,0.8417705,⋯,-0.005397045,0.01792956,1.792956,-0.009487555,-0.9487555,25.74938,85.83128,23.06883,76.89611,6


[1] "Carbon predicted"
